# Download AIA 1700 Å — 4 eventos de CME (mesmas datas da análise em 171 Å)

Réplica do pipeline de download usado para 171 Å, agora para o canal **1700 Å** do AIA/SDO, nos **4 eventos** usados na análise combinada de 171 Å (`Eclipse/02-Notebooks/main-fft-pca-sun-signals.ipynb`, bloco de PCA/FFT com `ruidos = [ruido, ruido_2, ruido_3, ruido_4]`):

| Data | Observação |
|---|---|
| 2011-06-05 | Halo, classe GOES B durante todo o período, sem *flares* associados (evento de referência principal) |
| 2017-04-24 | Evento de CME |
| 2017-04-30 | CME com *flare* associado, **não** halo |
| 2022-10-01 | CME com *flare* associado, halo |

**Por que 1700 Å?**
- **171 Å**: EUV, traça plasma coronal a ~1 MK → *dimming* coronal bem definido.
- **1700 Å**: UV próximo, traça cromosfera baixa / região de transição → comprimento de onda mais próximo do filtro UVM2 (2310 Å) do XMM-Newton/OM usado nas observações de HD 189733 A.

**Sobre as janelas temporais:** o notebook de download de 171 Å atual (`Sun/sdo_aia_download/main-sun-lightcurves-sunpy.ipynb`) já foi editado várias vezes e não contém mais as células de busca originais para todos os 4 eventos (por exemplo, não há mais uma célula de busca para 2017-04-24). Para garantir a **mesma cobertura temporal exata** em 1700 Å, as janelas abaixo foram reconstruídas diretamente a partir dos timestamps dos arquivos `.fits` de 171 Å já baixados em `Sun/sdo_aia_download/171/<data>[-no-cme]/`, não das células (potencialmente desatualizadas) do notebook.

**Nota:** a janela SEM CME de 2017-04-30 (01:00–01:50 UT) fica dentro da janela COM CME (00:10–06:15 UT) do mesmo dia — isso é herdado da escolha original dos dados em 171 Å, não uma decisão nova feita aqui.

**Saída:** os `.fits` são salvos em `Sun/sdo_aia_download/1700/<data>-1700` e `Sun/sdo_aia_download/1700/<data>-no-cme-1700`, seguindo a organização por subpasta de comprimento de onda adotada no repositório (`171/`, `1700/`, `304/`, `transit-venus/`).

In [1]:
import os

import matplotlib.pyplot as plt
import sunpy.map
from astropy import units as u
from sunpy.net import Fido, attrs as a


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Eventos e janelas temporais (reconstruídas a partir dos `.fits` de 171 Å já baixados)

In [2]:
WAVELENGTH = 1700 * u.angstrom
SAMPLE = (10 * 60) * u.s  # one frame every 10 minutes, same cadence as the 171 A downloads

# Time windows reconstructed from the timestamps of the existing 171 A fits files in
# Sun/sdo_aia_download/<date>[-no-cme]/, so the 1700 A coverage matches 171 A exactly.
EVENTS = [
    {
        "date": "2011-06-05",
        "note": "Halo CME, GOES class B throughout, no associated flares (main reference event)",
        "cme_window": ("2011-06-05 04:00", "2011-06-05 08:00"),
        "no_cme_window": ("2011-06-05 01:00", "2011-06-05 03:00"),
    },
    {
        "date": "2017-04-24",
        "note": "CME event",
        "cme_window": ("2017-04-24 00:10", "2017-04-24 03:55"),
        "no_cme_window": ("2017-04-24 04:00", "2017-04-24 06:15"),
    },
    {
        "date": "2017-04-30",
        "note": "CME with associated flare, not halo",
        "cme_window": ("2017-04-30 00:10", "2017-04-30 06:15"),
        "no_cme_window": ("2017-04-30 01:00", "2017-04-30 01:50"),
    },
    {
        "date": "2022-10-01",
        "note": "CME with associated flare, halo",
        "cme_window": ("2022-10-01 11:00", "2022-10-01 15:00"),
        "no_cme_window": ("2022-10-01 02:30", "2022-10-01 05:30"),
    },
]


## Função reutilizável de busca e download

In [3]:
def download_aia_frames(time_start, time_end, wavelength, sample, output_dir, max_retries=5):
    """
    Search AIA frames via SunPy Fido and download them to output_dir.
    Mirrors the search/fetch pattern used in the 171 A download notebook.

    The AIA export server (sdo7.nascom.nasa.gov) is often slow to serve
    individual files and times out under concurrent load, so failed files
    are retried (SunPy's own recommended pattern: pass the Results object
    back into Fido.fetch) with low concurrency instead of failing outright.
    """
    search_result = Fido.search(
        a.Time(time_start, time_end),
        a.Instrument("AIA"),
        a.Wavelength(wavelength),
        a.Sample(sample),
    )
    print(search_result)

    os.makedirs(output_dir, exist_ok=True)
    downloaded_files = Fido.fetch(search_result[0], path=output_dir, max_conn=2)

    retries = 0
    while downloaded_files.errors and retries < max_retries:
        retries += 1
        print(f"{len(downloaded_files.errors)} file(s) failed, retrying ({retries}/{max_retries})...")
        downloaded_files = Fido.fetch(downloaded_files, max_conn=2)

    if downloaded_files.errors:
        print(f"Still {len(downloaded_files.errors)} file(s) failing after {max_retries} retries:")
        print(downloaded_files.errors)
    else:
        print("Download errors: none")

    return downloaded_files


## Download — todos os 4 eventos (COM CME e SEM CME)

In [ ]:
for event in EVENTS:
    date = event["date"]
    cme_dir = os.path.join("..", "Sun", "sdo_aia_download", "1700", f"{date}-1700")
    no_cme_dir = os.path.join("..", "Sun", "sdo_aia_download", "1700", f"{date}-no-cme-1700")

    print(f"\n=== {date}: {event['note']} ===")

    print("-- with CME --")
    event["files_cme"] = download_aia_frames(
        *event["cme_window"], WAVELENGTH, SAMPLE, cme_dir
    )
    print(f"{len(event['files_cme'])} frames downloaded to {cme_dir}")

    print("-- without CME (reference) --")
    event["files_no_cme"] = download_aia_frames(
        *event["no_cme_window"], WAVELENGTH, SAMPLE, no_cme_dir
    )
    print(f"{len(event['files_no_cme'])} frames downloaded to {no_cme_dir}")


## Conferência rápida (quicklook)

Plota o primeiro frame COM CME de cada evento para checagem visual, salvando os resultados em `Eclipse/03-Products/1700A/` (dpi=300), seguindo o padrão de outputs do repositório ECLIPSE.

In [ ]:
def plot_quicklook(fits_path, title, fig_name,
                    output_dir=os.path.join("..", "Eclipse", "03-Products", "1700A")):
    """Quick-look plot of a single AIA 1700 A frame, saved at 300 dpi."""
    aia_map = sunpy.map.Map(fits_path)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(
        aia_map.data, origin="lower", cmap=aia_map.cmap,
        vmin=0, vmax=aia_map.data.max() * 0.3,
    )
    ax.set_title(title)
    ax.set_xlabel("Pixel X")
    ax.set_ylabel("Pixel Y")
    plt.colorbar(ax.images[0], ax=ax, label="Intensity")
    plt.tight_layout()

    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{fig_name}.png")
    plt.savefig(file_path, dpi=300, bbox_inches="tight")
    print(f"Quicklook saved to: {file_path}")
    plt.show()

    return aia_map


for event in EVENTS:
    date = event["date"]
    plot_quicklook(
        event["files_cme"][0],
        f"AIA 1700 A - with CME - {date}",
        f"quicklook_1700A_{date}_with-cme",
    )


## Próximos passos

Os `.fits` baixados aqui (`Sun/sdo_aia_download/1700/<data>-1700` e `Sun/sdo_aia_download/1700/<data>-no-cme-1700`, para os 4 eventos) são consumidos pelo notebook de análise **`main-fft-analysis-1700A.ipynb`**, que simula o trânsito de HD 189733 Ab sobre esses frames usando a classe `Estrela(useFits=True, fits_path=...)` do Core do ECLIPSE, e reproduz a análise combinada (curvas de luz, resíduo, PCA e FFT entre os 4 eventos) — exatamente como feito para 171 Å em `main-fft-pca-sun-signals.ipynb`.